## Loading CIFAR-10

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src").is_dir())))
from src.paths import ROOT


In [ ]:
import numpy as np
import tensorflow as tf

SEED = 42
IMG_SIZE = 160
BATCH_SIZE = 64
VAL_SIZE = 5_000

USE_AUGMENTATION = False

tf.keras.utils.set_random_seed(SEED)

# Load CIFAR-10 (downloaded automatically on the first run)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

# Keep images in [0, 255] (uint8) — each model’s preprocess_input handles scaling
# Squeeze labels to shape (N,) with dtype int
y_train = y_train.squeeze().astype("int64")
y_test = y_test.squeeze().astype("int64")

print("x_train:", x_train.shape, x_train.dtype, "min/max:", x_train.min(), x_train.max())
print("y_train:", y_train.shape, y_train.dtype, "classes:", np.unique(y_train).size)
print("x_test :", x_test.shape, x_test.dtype)
print("y_test :", y_test.shape, y_test.dtype)

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]
print("Example label:", int(y_train[0]), "->", class_names[int(y_train[0])])



## Dataset preparation (train/val/test)


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# Split train into train/val (last VAL_SIZE examples = validation)
x_val = x_train[-VAL_SIZE:]
y_val = y_train[-VAL_SIZE:]
x_tr = x_train[:-VAL_SIZE]
y_tr = y_train[:-VAL_SIZE]

print("train:", x_tr.shape, y_tr.shape)
print("val  :", x_val.shape, y_val.shape)
print("test :", x_test.shape, y_test.shape)


def preprocess_image(image, label, training: bool):
    # uint8 [0,255] -> float32
    image = tf.cast(image, tf.float32)

    if training and USE_AUGMENTATION:
        # Classic CIFAR augmentation: pad + random crop + flip
        image = tf.image.resize_with_crop_or_pad(image, 40, 40)
        image = tf.image.random_crop(image, size=[32, 32, 3])
        image = tf.image.random_flip_left_right(image)

    # Resize to backbone size (keeps a clean graph for TFLite)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), method="bilinear")

    return image, label


def make_ds(x, y, training: bool):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(10_000, seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(lambda im, lab: preprocess_image(im, lab, training), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds


train_ds = make_ds(x_tr, y_tr, training=True)
val_ds = make_ds(x_val, y_val, training=False)
test_ds = make_ds(x_test, y_test, training=False)



## Common settings (all models)

This cell holds everything shared across models (augmentation, callbacks, hyperparameters, helpers). To try a new model (e.g. MCU‑Net), add a `build_*()` and call `train_head_and_finetune(...)`.


In [ ]:
from pathlib import Path
from tensorflow.keras import layers

# The dataset already yields (IMG_SIZE, IMG_SIZE, 3) so the model input is fixed.
INPUT_SHAPE = (IMG_SIZE, IMG_SIZE, 3)
NUM_CLASSES = 10

# Training (shared)
EPOCHS_HEAD = 40
EPOCHS_FINE = 20
HEAD_LR = 1e-3
FINE_LR = 1e-5
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
FINE_TUNE_RATIO = 0.8

CHECKPOINT_DIR = ROOT / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

base_callbacks = [
    tf.keras.callbacks.TerminateOnNaN(),
    # Patience > ReduceLROnPlateau.patience to give time for LR to drop before stopping
    tf.keras.callbacks.EarlyStopping(
        patience=8,
        restore_best_weights=True,
        monitor="val_accuracy",
        mode="max",
        min_delta=1e-3,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        patience=3,
        factor=0.5,
        min_lr=1e-6,
        monitor="val_loss",
        verbose=1,
    ),
]


def make_common_stem(preprocess_layer: tf.keras.layers.Layer):
    """Build Input + a preprocess layer for the model.

    Prefer using **Keras layers** (not a Python `Lambda`) so the model is
    easy to save/load as `.keras` without extra `custom_objects`.
    """
    inputs = tf.keras.Input(shape=INPUT_SHAPE)
    x = preprocess_layer(inputs)
    return inputs, x


def make_sparse_ce_with_label_smoothing(label_smoothing: float):
    """Label smoothing for sparse (int) labels.

    Some versions don’t support `label_smoothing` in SparseCategoricalCrossentropy.
    Here we implement smoothing converting y_true -> one-hot dentro da loss.
    """

    if not label_smoothing or label_smoothing <= 0.0:
        return tf.keras.losses.SparseCategoricalCrossentropy()

    cce = tf.keras.losses.CategoricalCrossentropy()

    def loss(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_true = tf.one_hot(y_true, depth=NUM_CLASSES)
        y_true = y_true * (1.0 - label_smoothing) + (label_smoothing / NUM_CLASSES)
        return cce(y_true, y_pred)

    return loss


def make_optimizer(lr: float, weight_decay: float):
    """AdamW when available; otherwise Adam."""

    AdamW = getattr(tf.keras.optimizers, "AdamW", None)
    if AdamW is not None:
        return AdamW(learning_rate=lr, weight_decay=weight_decay)
    return tf.keras.optimizers.Adam(learning_rate=lr)


KERAS_MODELS_DIR = ROOT / "keras_models"
KERAS_MODELS_DIR.mkdir(parents=True, exist_ok=True)


def save_best_as_keras(build_fn, results: dict, filename: str):
    """Reload the uncompiled model, load best weights, and save a `.keras` file.

    This avoids common training serialization issues (custom loss, etc.).
    """

    weights_path = results.get("best_weights")
    if not weights_path:
        raise ValueError("results does not have 'best_weights'. Run training first.")

    model, _ = build_fn()
    model.load_weights(weights_path)

    out_path = KERAS_MODELS_DIR / filename
    model.save(str(out_path))
    print("Model saved at:", out_path.resolve())
    return out_path, model


def predict_labels(model: tf.keras.Model, ds):
    """Retorna y_true, y_pred e a matriz de probabilidades."""

    y_true = []
    y_prob = []
    for xb, yb in ds:
        prob = model.predict(xb, verbose=0)
        y_prob.append(prob)
        y_true.append(yb.numpy())

    y_true = np.concatenate(y_true).astype("int64")
    y_prob = np.concatenate(y_prob)
    y_pred = y_prob.argmax(axis=1).astype("int64")
    return y_true, y_pred, y_prob


def metrics_from_confusion_matrix(cm: np.ndarray):
    """Classic metrics from the confusion matrix."""

    cm = cm.astype(np.int64)
    tp = np.diag(cm)
    support = cm.sum(axis=1)
    fp = cm.sum(axis=0) - tp
    fn = support - tp

    precision = tp / np.maximum(tp + fp, 1)
    recall = tp / np.maximum(tp + fn, 1)
    f1 = (2 * precision * recall) / np.maximum(precision + recall, 1e-12)
    per_class_acc = tp / np.maximum(support, 1)

    acc = tp.sum() / np.maximum(cm.sum(), 1)
    macro_f1 = f1.mean()
    weighted_f1 = (f1 * support).sum() / np.maximum(support.sum(), 1)

    return {
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "per_class_accuracy": per_class_acc,
        "per_class_f1": f1,
        "per_class_precision": precision,
        "per_class_recall": recall,
        "support": support,
    }


def plot_per_class_accuracy_heatmap(per_class_acc: np.ndarray, class_names: list[str], title: str):
    import matplotlib.pyplot as plt

    per_class_acc = np.asarray(per_class_acc)

    fig, ax = plt.subplots(figsize=(10, 1.6))
    im = ax.imshow(per_class_acc[None, :], aspect="auto", cmap="viridis", vmin=0, vmax=1)

    ax.set_yticks([0])
    ax.set_yticklabels(["acc"])
    ax.set_xticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")

    for i, v in enumerate(per_class_acc):
        ax.text(i, 0, f"{v:.2f}", ha="center", va="center", color="white" if v < 0.5 else "black")

    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    ax.set_title(title)
    fig.tight_layout()


def plot_confusion_matrix_heatmap(cm: np.ndarray, class_names: list[str], title: str, normalize: bool = True):
    import matplotlib.pyplot as plt

    cm = cm.astype(np.float32)
    if normalize:
        cm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, ax = plt.subplots(figsize=(7, 6.5))
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1 if normalize else cm.max())

    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right")
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            v = cm[i, j]
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if v > 0.5 else "black")

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()


def evaluate_model(model: tf.keras.Model, ds, class_names: list[str], title: str):
    y_true, y_pred, _ = predict_labels(model, ds)
    cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=len(class_names)).numpy()

    m = metrics_from_confusion_matrix(cm)
    print(f"[{title}] accuracy={m['accuracy']:.4f} | macro_f1={m['macro_f1']:.4f} | weighted_f1={m['weighted_f1']:.4f}")

    # Per-class
    for i, cname in enumerate(class_names):
        print(
            f"  {cname:>10s}: acc={m['per_class_accuracy'][i]:.3f} "
            f"f1={m['per_class_f1'][i]:.3f} support={int(m['support'][i])}"
        )

    plot_per_class_accuracy_heatmap(m["per_class_accuracy"], class_names, title=f"{title} — accuracy per class")
    plot_confusion_matrix_heatmap(cm, class_names, title=f"{title} — confusion matrix (normalizada)", normalize=True)
    return m, cm


def train_head_and_finetune(
    model: tf.keras.Model,
    base: tf.keras.Model | None,
    model_tag: str,
    head_epochs: int = EPOCHS_HEAD,
    fine_epochs: int = EPOCHS_FINE,
    head_lr: float = HEAD_LR,
    fine_lr: float = FINE_LR,
    weight_decay: float = WEIGHT_DECAY,
    label_smoothing: float = LABEL_SMOOTHING,
    fine_tune_ratio: float = FINE_TUNE_RATIO,
):
    """Train the head (frozen base) then fine-tune partially (if base is not None).

    - Uses AdamW + label smoothing
    - Keeps the best val_accuracy checkpoint (handy to export/convert later).
    """

    ckpt_path = CHECKPOINT_DIR / f"{model_tag.replace(' ', '_')}_best.weights.h5"
    ckpt_cb = tf.keras.callbacks.ModelCheckpoint(
        ckpt_path,
        monitor="val_accuracy",
        save_best_only=True,
        save_weights_only=True,
    )
    callbacks = base_callbacks + [ckpt_cb]

    loss_fn = make_sparse_ce_with_label_smoothing(label_smoothing)
    metrics = [
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5"),
    ]

    # Head
    model.compile(
        optimizer=make_optimizer(head_lr, weight_decay),
        loss=loss_fn,
        metrics=metrics,
    )

    history_head = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=head_epochs,
        callbacks=callbacks,
    )

    # Fine-tune (2nd training phase when base=None)
    history_ft = None
    if fine_epochs > 0:
        if base is not None:
            base.trainable = True

            fine_tune_at = int(len(base.layers) * fine_tune_ratio)
            for layer in base.layers[:fine_tune_at]:
                layer.trainable = False

            # Keep BatchNorm in inference mode during fine-tuning (common practice)
            for layer in base.layers[fine_tune_at:]:
                if isinstance(layer, layers.BatchNormalization):
                    layer.trainable = False

        model.compile(
            optimizer=make_optimizer(fine_lr, weight_decay),
            loss=loss_fn,
            metrics=metrics,
        )

        history_ft = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=fine_epochs,
            callbacks=callbacks,
        )

    # Final evaluation with the best checkpoint weights (before early stopping restores them)
    model.load_weights(ckpt_path)
    loss_test, acc_test, top5_test = model.evaluate(test_ds, verbose=0)

    print(f"[{model_tag}] Test acc (best weights): {acc_test:.4f} | top5: {top5_test:.4f} | loss: {loss_test:.4f}")

    return {
        "test_acc_head": float("nan"),
        "test_loss_head": float("nan"),
        "test_acc_finetune": float(acc_test),
        "test_loss_finetune": float(loss_test),
        "test_top5": float(top5_test),
        "best_weights": str(ckpt_path),
    }, history_head, history_ft



## MobileNetV3-Small

In [ ]:
from tensorflow.keras.applications import MobileNetV3Small


def build_mobilenetv3_small(num_classes: int = NUM_CLASSES, dropout: float = 0.2):
    # Dataset output is float32 in [0, 255]
    preprocess = layers.Rescaling(1.0 / 127.5, offset=-1.0, name="preprocess_mnv3")
    inputs, x = make_common_stem(preprocess)

    base = MobileNetV3Small(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_preprocessing=False,  # we already rescale to (-1, 1) above
    )
    base.trainable = False

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs, name="MobileNetV3Small_CIFAR10")
    return model, base


mobilenet_model, mobilenet_base = build_mobilenetv3_small()
mnv3_results, history_mnv3_head, history_mnv3_ft = train_head_and_finetune(
    mobilenet_model,
    mobilenet_base,
    model_tag="MobileNetV3-Small",
)

In [ ]:
# Save the best model as .keras and evaluate (accuracy, F1, heatmaps)
mnv3_keras_path, mnv3_keras_model = save_best_as_keras(
    build_mobilenetv3_small,
    mnv3_results,
    filename="MobileNetV3Small_CIFAR10.keras",
)

In [ ]:
mnv3_eval, mnv3_cm = evaluate_model(mnv3_keras_model, test_ds, class_names, title="MobileNetV3-Small")


## EfficientNetB0


In [ ]:
import inspect
from tensorflow.keras.applications import EfficientNetB0


def build_efficientnetb0(num_classes: int = NUM_CLASSES, dropout: float = 0.3):
    preprocess = layers.Identity(name="preprocess_effnet_identity")
    inputs, x = make_common_stem(preprocess)

    eff_kwargs = dict(
        include_top=False,
        weights="imagenet",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    # Some versions of EfficientNetB0 have a built-in preprocessing layer that we want to disable
    if "include_preprocessing" in inspect.signature(EfficientNetB0).parameters:
        eff_kwargs["include_preprocessing"] = False

    base = EfficientNetB0(**eff_kwargs)
    base.trainable = False

    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs, name="EfficientNetB0_CIFAR10")
    return model, base


effnet_model, effnet_base = build_efficientnetb0()

effnet_results, history_eff_head, history_eff_ft = train_head_and_finetune(
    effnet_model,
    effnet_base,
    model_tag="EfficientNetB0",
)


In [ ]:
# Save the best model as .keras and evaluate (accuracy, F1, heatmaps)
effb0_keras_path, effb0_keras_model = save_best_as_keras(
    build_efficientnetb0,
    effnet_results,
    filename="EfficientNetB0_CIFAR10.keras",
)

In [ ]:
effb0_eval, effb0_cm = evaluate_model(effb0_keras_model, test_ds, class_names, title="EfficientNetB0")

## Export to SavedModel (for later conversion to `.tflite`)

After a kernel restart, variables like `mnv3_results` may be gone — this export path uses **file paths** only.

You can export in two ways:

- **From `.keras`** (recommended): `keras_models/<name>.keras`
- **From weights**: `checkpoints/<tag>_best.weights.h5` (rebuilds the model and loads the weights)

The result is a **SavedModel** in `saved_models/`, ready for `TFLiteConverter`.

In [ ]:
import shutil
from pathlib import Path

SAVEDMODEL_DIR = ROOT / "saved_models"
SAVEDMODEL_DIR.mkdir(parents=True, exist_ok=True)


def export_savedmodel_from_keras(keras_path: str, export_name: str):
    model = tf.keras.models.load_model(keras_path)

    out_dir = SAVEDMODEL_DIR / export_name
    if out_dir.exists():
        shutil.rmtree(out_dir)

    # Keras 3: preferir .export(); fallback para tf.saved_model.save
    if hasattr(model, "export"):
        model.export(str(out_dir))
    else:
        tf.saved_model.save(model, str(out_dir))

    print("SavedModel exported at:", out_dir.resolve())
    return out_dir


models_to_export = [
    ("keras_models/MobileNetV3Small_CIFAR10.keras", "MobileNetV3Small_CIFAR10"),
    ("keras_models/EfficientNetB0_CIFAR10.keras", "EfficientNetB0_CIFAR10"),
    ("keras_models/MCUNet_CIFAR10.keras", "MCUNet_CIFAR10"),
]

for keras_path, name in models_to_export:
    if Path(keras_path).exists():
        export_savedmodel_from_keras(keras_path, name)
    else:
        print("File not found (skipping):", keras_path)



## Official MCUNet (mit-han-lab/mcunet)

We use the **official MCUNet** repo (`third_party/mcunet-official`), which provides:
- Pre-trained ImageNet models (PyTorch)
- Support for automatic downloading of configs and weights
- ProxylessNAS architecture optimized for microcontrollers

Workflow:
1. Load the pre-trained PyTorch model from ImageNet
2. Convert the weights to Keras
3. Fine-tune on CIFAR-10

In [ ]:
import sys
import json
import torch
import numpy as np
from pathlib import Path

MCUNET_OFFICIAL_PATH = Path("third_party/mcunet-official")
if str(MCUNET_OFFICIAL_PATH) not in sys.path:
    sys.path.insert(0, str(MCUNET_OFFICIAL_PATH))

from mcunet.model_zoo import build_model as build_mcunet_pytorch, net_id_list, download_tflite

# =====================================================================
# Custom ImageNet normalization layer (registered for serialization)
# =====================================================================
@tf.keras.utils.register_keras_serializable(package="MCUNet")
class ImageNetNormalization(layers.Layer):
    """ImageNet normalization: (x/255 - mean) / std"""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.mean = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
        self.std = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)
    
    def call(self, x):
        x = x / 255.0
        return (x - self.mean) / self.std
    
    def get_config(self):
        return super().get_config()

print("=" * 70)
print("MCUNet (official) — available models:")
print("=" * 70)
for net_id in net_id_list:
    print(f"  - {net_id}")

# =====================================================================
# MCUNet model choice
# =====================================================================
# Opções ImageNet: mcunet-in0 (10fps), mcunet-in1 (5fps), mcunet-in2, mcunet-in3, mcunet-in4
# mcunet-in4 (512KB SRAM, 2MB Flash) is the largest variant—best for apples-to-apples comparison
MCUNET_NET_ID = "mcunet-in4"

print(f"\nLoading model: {MCUNET_NET_ID}")
pt_model, resolution, description = build_mcunet_pytorch(MCUNET_NET_ID, pretrained=True)
print(f"Native resolution: {resolution}x{resolution}")
print(f"Description: {description}")
pt_model.eval()

# Show summary architecture
total_params = sum(p.numel() for p in pt_model.parameters())
trainable_params = sum(p.numel() for p in pt_model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
def build_mcunet_official(
    num_classes: int = NUM_CLASSES,
    dropout: float = 0.2,
    net_id: str = MCUNET_NET_ID,
):
    """
    Build a Keras model from the official MCUNet architecture.
    
    Load ImageNet pre-trained PyTorch weights and convert to Keras.
    Returns (full_model, backbone) for train_head_and_finetune.
    """
    # Load PyTorch model with pre-trained weights
    pt_model, resolution, _ = build_mcunet_pytorch(net_id, pretrained=True)
    pt_model.eval()
    pt_state = pt_model.state_dict()
    
    # Read network config
    net_config = pt_model.config
    
    # BatchNorm parameters
    bn_momentum = 0.9  # Keras default (PyTorch uses 0.1, which is 1-0.9)
    bn_eps = 1e-5
    
    # ========== BUILD KERAS MODEL ==========
    
    # Uses the ImageNetNormalization class registered externally
    preprocess = ImageNetNormalization(name="preprocess_mcunet_official")
    inputs, x = make_common_stem(preprocess)
    
    # Activation helper
    def _act(x, act_name, name):
        if act_name in (None, "none"):
            return x
        if act_name == "relu6":
            return layers.ReLU(max_value=6.0, name=name)(x)
        if act_name == "relu":
            return layers.ReLU(name=name)(x)
        return x
    
    # First conv
    first_conv_cfg = net_config.get("first_conv", {})
    out_ch = first_conv_cfg.get("out_channels", 32)
    ks = first_conv_cfg.get("kernel_size", 3)
    stride = first_conv_cfg.get("stride", 2)
    act = first_conv_cfg.get("act_func", "relu6")
    
    x = layers.Conv2D(out_ch, ks, strides=stride, padding="same", use_bias=False, 
                      name="mcunet_off_first_conv")(x)
    x = layers.BatchNormalization(momentum=bn_momentum, epsilon=bn_eps, 
                                  name="mcunet_off_first_bn")(x)
    x = _act(x, act, name="mcunet_off_first_act")
    
    # MBConv blocks
    blocks_cfg = net_config.get("blocks", [])
    for i, block in enumerate(blocks_cfg):
        mic = block.get("mobile_inverted_conv", {})
        if not mic or mic.get("name") == "ZeroLayer":
            continue
        
        in_ch = mic.get("in_channels", 0)
        out_ch = mic.get("out_channels", 0)
        k = mic.get("kernel_size", 3)
        s = mic.get("stride", 1)
        expand_ratio = mic.get("expand_ratio", 1)
        mid_ch = mic.get("mid_channels")
        mid_ch = mid_ch if mid_ch else int(in_ch * expand_ratio)
        act = mic.get("act_func", "relu6")
        use_se = mic.get("use_se", False)
        
        shortcut_cfg = block.get("shortcut")
        has_shortcut = shortcut_cfg is not None and s == 1 and in_ch == out_ch
        x_in = x
        
        # Expand (1x1)
        if expand_ratio != 1:
            x = layers.Conv2D(mid_ch, 1, padding="same", use_bias=False, 
                              name=f"mcunet_off_b{i}_expand")(x)
            x = layers.BatchNormalization(momentum=bn_momentum, epsilon=bn_eps, 
                                          name=f"mcunet_off_b{i}_expand_bn")(x)
            x = _act(x, act, name=f"mcunet_off_b{i}_expand_act")
        
        # Depthwise
        x = layers.DepthwiseConv2D(k, strides=s, padding="same", use_bias=False, 
                                    name=f"mcunet_off_b{i}_dw")(x)
        x = layers.BatchNormalization(momentum=bn_momentum, epsilon=bn_eps, 
                                      name=f"mcunet_off_b{i}_dw_bn")(x)
        x = _act(x, act, name=f"mcunet_off_b{i}_dw_act")
        
        # SE block (Squeeze-and-Excitation) if present
        if use_se:
            se_ch = max(1, mid_ch // 4)
            se = layers.GlobalAveragePooling2D(keepdims=True, name=f"mcunet_off_b{i}_se_pool")(x)
            se = layers.Conv2D(se_ch, 1, activation="relu", name=f"mcunet_off_b{i}_se_reduce")(se)
            se = layers.Conv2D(mid_ch, 1, activation="sigmoid", name=f"mcunet_off_b{i}_se_expand")(se)
            x = layers.Multiply(name=f"mcunet_off_b{i}_se_mult")([x, se])
        
        # Project (1x1)
        x = layers.Conv2D(out_ch, 1, padding="same", use_bias=False, 
                          name=f"mcunet_off_b{i}_project")(x)
        x = layers.BatchNormalization(momentum=bn_momentum, epsilon=bn_eps, 
                                      name=f"mcunet_off_b{i}_project_bn")(x)
        
        # Shortcut
        if has_shortcut:
            x = layers.Add(name=f"mcunet_off_b{i}_add")([x_in, x])
    
    # Feature conv (before classifier)
    feature_mix_cfg = net_config.get("feature_mix_layer", {})
    if feature_mix_cfg and feature_mix_cfg.get("out_channels"):
        fm_out = feature_mix_cfg["out_channels"]
        x = layers.Conv2D(fm_out, 1, padding="same", use_bias=False, 
                          name="mcunet_off_feature_mix")(x)
        x = layers.BatchNormalization(momentum=bn_momentum, epsilon=bn_eps, 
                                      name="mcunet_off_feature_mix_bn")(x)
        x = _act(x, feature_mix_cfg.get("act_func", "relu6"), name="mcunet_off_feature_mix_act")
    
    x = layers.GlobalAveragePooling2D(name="mcunet_off_gap")(x)
    
    # Backbone (without classifier)
    backbone = tf.keras.Model(inputs, x, name="MCUNet_Official_Backbone")
    
    # Head CIFAR-10
    x = layers.Dropout(dropout, name="mcunet_off_dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="mcunet_off_classifier")(x)
    
    model = tf.keras.Model(inputs, outputs, name="MCUNet_Official_CIFAR10")
    
    # ========== LOAD PYTORCH WEIGHTS ==========
    print(f"Converting PyTorch weights -> Keras...")
    loaded_count = 0
    
    for keras_layer in model.layers:
        layer_name = keras_layer.name
        weights_to_set = []
        
        try:
            # First conv
            if layer_name == "mcunet_off_first_conv":
                w = pt_state.get("first_conv.conv.weight")
                if w is not None:
                    weights_to_set = [w.numpy().transpose(2, 3, 1, 0)]
                    
            elif layer_name == "mcunet_off_first_bn":
                g = pt_state.get("first_conv.bn.weight")
                b = pt_state.get("first_conv.bn.bias")
                m = pt_state.get("first_conv.bn.running_mean")
                v = pt_state.get("first_conv.bn.running_var")
                if g is not None:
                    weights_to_set = [g.numpy(), b.numpy(), m.numpy(), v.numpy()]
            
            # Blocks - expand
            elif "_expand" in layer_name and "_bn" not in layer_name and "_act" not in layer_name:
                idx = layer_name.split("_b")[1].split("_")[0]
                w = pt_state.get(f"blocks.{idx}.mobile_inverted_conv.inverted_bottleneck.conv.weight")
                if w is not None:
                    weights_to_set = [w.numpy().transpose(2, 3, 1, 0)]
                    
            elif "_expand_bn" in layer_name:
                idx = layer_name.split("_b")[1].split("_")[0]
                base = f"blocks.{idx}.mobile_inverted_conv.inverted_bottleneck.bn"
                g = pt_state.get(f"{base}.weight")
                if g is not None:
                    weights_to_set = [g.numpy(), 
                                      pt_state[f"{base}.bias"].numpy(),
                                      pt_state[f"{base}.running_mean"].numpy(),
                                      pt_state[f"{base}.running_var"].numpy()]
            
            # Blocks - depthwise
            elif "_dw" in layer_name and "_bn" not in layer_name and "_act" not in layer_name:
                idx = layer_name.split("_b")[1].split("_")[0]
                w = pt_state.get(f"blocks.{idx}.mobile_inverted_conv.depth_conv.conv.weight")
                if w is not None:
                    weights_to_set = [w.numpy().transpose(2, 3, 0, 1)]
                    
            elif "_dw_bn" in layer_name:
                idx = layer_name.split("_b")[1].split("_")[0]
                base = f"blocks.{idx}.mobile_inverted_conv.depth_conv.bn"
                g = pt_state.get(f"{base}.weight")
                if g is not None:
                    weights_to_set = [g.numpy(),
                                      pt_state[f"{base}.bias"].numpy(),
                                      pt_state[f"{base}.running_mean"].numpy(),
                                      pt_state[f"{base}.running_var"].numpy()]
            
            # Blocks - project
            elif "_project" in layer_name and "_bn" not in layer_name:
                idx = layer_name.split("_b")[1].split("_")[0]
                w = pt_state.get(f"blocks.{idx}.mobile_inverted_conv.point_linear.conv.weight")
                if w is not None:
                    weights_to_set = [w.numpy().transpose(2, 3, 1, 0)]
                    
            elif "_project_bn" in layer_name:
                idx = layer_name.split("_b")[1].split("_")[0]
                base = f"blocks.{idx}.mobile_inverted_conv.point_linear.bn"
                g = pt_state.get(f"{base}.weight")
                if g is not None:
                    weights_to_set = [g.numpy(),
                                      pt_state[f"{base}.bias"].numpy(),
                                      pt_state[f"{base}.running_mean"].numpy(),
                                      pt_state[f"{base}.running_var"].numpy()]
            
            # Feature mix layer
            elif layer_name == "mcunet_off_feature_mix":
                w = pt_state.get("feature_mix_layer.conv.weight")
                if w is not None:
                    weights_to_set = [w.numpy().transpose(2, 3, 1, 0)]
                    
            elif layer_name == "mcunet_off_feature_mix_bn":
                base = "feature_mix_layer.bn"
                g = pt_state.get(f"{base}.weight")
                if g is not None:
                    weights_to_set = [g.numpy(),
                                      pt_state[f"{base}.bias"].numpy(),
                                      pt_state[f"{base}.running_mean"].numpy(),
                                      pt_state[f"{base}.running_var"].numpy()]
            
            if weights_to_set:
                keras_layer.set_weights(weights_to_set)
                loaded_count += 1
                
        except Exception as e:
            pass  # ignore unmapped layer errors
    
    print(f"✓ Weights loaded: {loaded_count} layers")
    
    # Freeze backbone to train the head only
    for layer in backbone.layers:
        layer.trainable = False
    
    return model, backbone


# =====================================================================
# Build and train Official MCUNet
# =====================================================================
print("\n" + "=" * 70)
print("Training Official MCUNet on CIFAR-10")
print("=" * 70)

mcunet_off_model, mcunet_off_base = build_mcunet_official()

mcunet_off_results, history_mcunet_off_head, history_mcunet_off_ft = train_head_and_finetune(
    mcunet_off_model,
    mcunet_off_base,
    model_tag="MCUNet_Official",
)

In [ ]:
# Save and evaluate MCUNet
mcunet_off_keras_path, mcunet_off_keras_model = save_best_as_keras(
    build_mcunet_official,
    mcunet_off_results,
    filename="MCUNet_CIFAR10.keras",
)

mcunet_off_eval, mcunet_off_cm = evaluate_model(
    mcunet_off_keras_model, test_ds, class_names, title="MCUNet Official"
)

## Expoorting the test set (for Android/Raspberry)
Here we store the same **`x_test`/`y_test`** in portable formats for other devices:

- **`cifar10_test_uint8.npz`**: Use on Raspberry Pi
- **`images_png/` + `labels.csv`**: Easy to consume in apps (e.g., Android), using any PNG reader

Tip: on Android, export only a **subset** (e.g. 500–2000 images) to keep the app size small.


In [ ]:
import csv
from pathlib import Path

EXPORT_DIR = ROOT / "exports" / "cifar10"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# How many images to export.
# - None: export everything (10k)
# - 1000/2000: lighter for Android
EXPORT_LIMIT = 2000

n = len(x_test) if EXPORT_LIMIT is None else int(min(EXPORT_LIMIT, len(x_test)))
print("Exporting", n, "test images to", EXPORT_DIR.resolve())

# 1) NPZ (Python/Raspberry)
npz_path = EXPORT_DIR / "cifar10_test_uint8.npz"
np.savez_compressed(
    npz_path,
    x_test=x_test[:n],
    y_test=y_test[:n],
    class_names=np.array(class_names),
)
print("OK:", npz_path)

# 2) PNG + labels.csv (Android/other)
img_dir = EXPORT_DIR / "images_png"
img_dir.mkdir(parents=True, exist_ok=True)

labels_path = EXPORT_DIR / "labels.csv"
with labels_path.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["filename", "label", "class_name"])
    for i in range(n):
        label = int(y_test[i])
        fname = f"{i:05d}_label{label}.png"
        out_path = img_dir / fname

        # x_test[i] is uint8 (0..255) shape (32,32,3)
        png_bytes = tf.io.encode_png(x_test[i]).numpy()
        tf.io.write_file(str(out_path), png_bytes)

        w.writerow([fname, label, class_names[label]])

print("OK:", img_dir, "(PNGs)")
print("OK:", labels_path)

# How to copy to devices:
# - Raspberry deployment (anonymized)
# - Android (adb):   adb push export_cifar10 /sdcard/Download/


In [ ]:
import tensorflow as tf

tflite_models_dir = ROOT / "tflite_models"
tflite_models_dir.mkdir(exist_ok=True)

print("Converting .keras models to .tflite:")
for keras_file in model_files:
    model = tf.keras.models.load_model(keras_file)
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()
    tflite_file = tflite_models_dir / (keras_file.stem + ".tflite")
    with open(tflite_file, "wb") as f:
        f.write(tflite_model)
    size_mb = os.path.getsize(tflite_file) / (1024*1024)
    print(f"{keras_file.name:35s} -> {tflite_file.name:35s} {size_mb:8.2f} MB")
print("Conversion completed. Models saved in:", tflite_models_dir)
